<a href="https://colab.research.google.com/github/mimomaina/Career-Path-Recommendation-System/blob/main/FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 65.0 MB/s eta 0:00:00


In [6]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [7]:
# Load dataset with low_memory=False to avoid DtypeWarning
df = pd.read_csv("/content/tech_industry_dataset.csv", low_memory=False)

job_embeddings =np.load("/content/job_embeddings_full.npy")

# Verify shapes before proceeding
print(f"Dataset Shape: {df.shape}")
print(f"Embeddings Shape: {job_embeddings.shape}")

df = df.iloc[:job_embeddings.shape[0]].reset_index(drop=True)


Dataset Shape: (292167, 22)
Embeddings Shape: (292167, 384)


In [8]:
from sklearn.cluster import MiniBatchKMeans

# Define & Train MiniBatch K-Means Model
num_clusters = 30
kmeans = MiniBatchKMeans(n_clusters=num_clusters, batch_size=10000, random_state=42)

df["Cluster"] = kmeans.fit_predict(job_embeddings)

# Display Sample Clustered Jobs
print("Job clustering complete. Sample clusters:")
print(df.groupby("Cluster")["Job Title"].unique().head())

# Save the Clustered Dataset
df.to_csv("clustered_jobs.csv", index=False)

# Save Cluster Centroids for Future Use
np.save("cluster_centroids.npy", kmeans.cluster_centers_)


Job clustering complete. Sample clusters:
Cluster
0         [Data Analyst, Data Scientist]
1    [Web Designer, Front-End Developer]
2    [Network Administrator, IT Manager]
3     [Software Tester, Systems Analyst]
4                      [Network Analyst]
Name: Job Title, dtype: object


In [11]:
# Step 1: Normalize embeddings for cosine similarity
job_embeddings = job_embeddings / np.linalg.norm(job_embeddings, axis=1, keepdims=True)

# Step 2: Create a FAISS index for cosine similarity
dimension = job_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(job_embeddings)

from sentence_transformers import SentenceTransformer

# Load the SBERT model
model = SentenceTransformer("all-MiniLM-L6-v2")  # Load model


# Step 3: Encode user input with weighted skills
weighted_skills = "Python + Python + Machine Learning + SQL"
user_embedding = model.encode(weighted_skills)
user_embedding = user_embedding.astype(np.float32).reshape(1, -1)
user_embedding = user_embedding / np.linalg.norm(user_embedding, axis=1, keepdims=True)

# Step 4: Perform similarity search
distances, indices = index.search(user_embedding, k=10)

# Step 5: Filter results for diversity
unique_titles = set()
diverse_recommendations = []
for idx in indices.flatten():
    title = df.iloc[idx]["Job Title"]
    if title not in unique_titles:
        diverse_recommendations.append(df.iloc[idx])
        unique_titles.add(title)
    if len(diverse_recommendations) == 5:  # Top 5 diverse results
        break

# Display diverse recommendations
print("Diverse Recommendations:")
for job in diverse_recommendations:
    print(job["Job Title"], job["Cluster"])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Diverse Recommendations:
Data Scientist 0


In [12]:
#Save the FAISS Index to Disk
faiss.write_index(index, "faiss_index.index")
print("FAISS index saved to 'faiss_index.index'.")

#Load the FAISS Index from Disk
loaded_index = faiss.read_index("faiss_index.index")
print(f"FAISS index loaded. Indexed {loaded_index.ntotal} job embeddings.")

FAISS index saved to 'faiss_index.index'.
FAISS index loaded. Indexed 292167 job embeddings.


In [13]:
print("Recommended Role:", job["Job Title"])
print("Cluster:", job["Cluster"])
print("Required Skills:", job["skills"])  # Assuming "Skills" column exists

Recommended Role: Data Scientist
Cluster: 0
Required Skills: Machine learning algorithms Python programming Data preprocessing Deep learning Model evaluation


In [14]:
required_skills = set(job["skills"].split(", "))
user_skills = set(["Python", "SQL"])
missing_skills = required_skills - user_skills
print(f"Missing Skills: {missing_skills}")

Missing Skills: {'Machine learning algorithms Python programming Data preprocessing Deep learning Model evaluation'}


In [15]:
# Define test cases (user inputs for different roles)
test_cases = {
    "Data Scientist": "Python, Machine Learning, SQL",
    "Software Engineer": "Python, Java, Software Development, Algorithms",
    "DevOps Engineer": "AWS, Kubernetes, CI/CD, Docker",
    "UX Designer": "User Experience, Wireframing, Prototyping, Figma",
    "Data Engineer": "SQL, ETL, Big Data, Apache Spark",
    "Product Manager": "Agile, Roadmapping, Stakeholder Management, Product Lifecycle"
}

# Loop through each test case and generate recommendations
for role, skills in test_cases.items():
    print(f"\nTesting for Role: {role}")
    print(f"Skills: {skills}")

    # Encode user input
    user_embedding = model.encode(skills)
    user_embedding = user_embedding.astype(np.float32).reshape(1, -1)
    user_embedding = user_embedding / np.linalg.norm(user_embedding, axis=1, keepdims=True)

    # Perform similarity search
    distances, indices = index.search(user_embedding, k=10)

    # Filter results for diversity
    unique_titles = set()
    diverse_recommendations = []
    for idx in indices.flatten():
        title = df.iloc[idx]["Job Title"]
        if title not in unique_titles:
            diverse_recommendations.append((title, df.iloc[idx]["Cluster"]))
            unique_titles.add(title)
        if len(diverse_recommendations) == 5:  # Top 5 diverse results
            break

    # Display results
    print("Diverse Recommendations:")
    for job_title, cluster in diverse_recommendations:
        print(f"- {job_title} (Cluster {cluster})")


Testing for Role: Data Scientist
Skills: Python, Machine Learning, SQL
Diverse Recommendations:
- Data Analyst (Cluster 0)

Testing for Role: Software Engineer
Skills: Python, Java, Software Development, Algorithms
Diverse Recommendations:
- Software Developer (Cluster 25)

Testing for Role: DevOps Engineer
Skills: AWS, Kubernetes, CI/CD, Docker
Diverse Recommendations:
- Systems Engineer (Cluster 20)

Testing for Role: UX Designer
Skills: User Experience, Wireframing, Prototyping, Figma
Diverse Recommendations:
- Front-End Developer (Cluster 1)

Testing for Role: Data Engineer
Skills: SQL, ETL, Big Data, Apache Spark
Diverse Recommendations:
- Data Engineer (Cluster 17)

Testing for Role: Product Manager
Skills: Agile, Roadmapping, Stakeholder Management, Product Lifecycle
Diverse Recommendations:
- Network Administrator (Cluster 2)


In [17]:
# Define test cases (user inputs for different roles)
test_cases = {
    "Data Scientist": "Python, Machine Learning, SQL",
    "Software Engineer": "Python, Java, Software Development, Algorithms",
    "DevOps Engineer": "AWS, Kubernetes, CI/CD, Docker",
    "UX Designer": "User Experience, Wireframing, Prototyping, Figma",
    "Data Engineer": "SQL, ETL, Big Data, Apache Spark",
    "Product Manager": "Agile, Roadmapping, Stakeholder Management, Product Lifecycle"
}

# Loop through each test case and generate recommendations
for role, skills in test_cases.items():
    print(f"\nTesting for Role: {role}")
    print(f"Skills: {skills}")

    # Encode user input
    user_embedding = model.encode(skills)
    user_embedding = user_embedding.astype(np.float32).reshape(1, -1)
    user_embedding = user_embedding / np.linalg.norm(user_embedding, axis=1, keepdims=True)

    # Perform similarity search
    distances, indices = index.search(user_embedding, k=10)

    # Filter results for diversity
    unique_titles = set()
    diverse_recommendations = []
    for idx in indices.flatten():
        title = df.iloc[idx]["Job Title"]
        if title not in unique_titles:
            diverse_recommendations.append((title, df.iloc[idx]["Cluster"], df.iloc[idx]["skills"]))
            unique_titles.add(title)
        if len(diverse_recommendations) == 5:  # Top 5 diverse results
            break

    # Display results
    print("Diverse Recommendations:")
    for job_title, cluster, required_skills in diverse_recommendations:
        # Convert required skills and user skills to sets
        required_skills_set = set(required_skills.split(", "))
        user_skills_set = set(skills.split(", "))

        # Calculate missing skills
        missing_skills = required_skills_set - user_skills_set

        # Display recommendation and missing skills
        print(f"- {job_title} (Cluster {cluster})")
        # print(f"  Required Skills: {', '.join(required_skills_set)}")
        print(f"  Missing Skills: {', '.join(missing_skills) if missing_skills else 'None'}")


Testing for Role: Data Scientist
Skills: Python, Machine Learning, SQL
Diverse Recommendations:
- Data Analyst (Cluster 0)
  Missing Skills: scikit-learn, PyTorch) Statistical analysis and modeling Data preprocessing and cleaning Big data technologies (e.g., R), Hadoop, Machine learning algorithms and libraries (e.g., Spark) Data visualization Strong programming skills (Python, TensorFlow

Testing for Role: Software Engineer
Skills: Python, Java, Software Development, Algorithms
Diverse Recommendations:
- Software Developer (Cluster 25)
  Missing Skills: Swift, Kotlin) Cross-platform development (e.g., React Native, Mobile app development languages (e.g., Flutter) Mobile app design principles APIs and web services integration Debugging and troubleshooting

Testing for Role: DevOps Engineer
Skills: AWS, Kubernetes, CI/CD, Docker
Diverse Recommendations:
- Systems Engineer (Cluster 20)
  Missing Skills: Cloud systems engineering Cloud infrastructure (e.g., Azure) DevOps practices Automa

In [20]:
# Step 4: Define a function to get the most similar job
def get_most_similar_job(query_text):
    """
    Perform a similarity search using FAISS and return the single most similar job.

    Args:
        query_text (str): The user input text (e.g., skills or job description).

    Returns:
        dict: A dictionary containing the most similar job's details and similarity score.
    """
    # Encode the query text into an embedding
    query_embedding = model.encode(query_text)
    query_embedding = query_embedding.astype(np.float32).reshape(1, -1)

    # Perform similarity search using FAISS
    distances, indices = index.search(query_embedding, k=2)  # Retrieve top 2 to exclude self-match

    # Exclude the exact match (if the query job exists in the dataset)
    most_similar_idx = indices.flatten()[1]  # Skip the first result (self-match)

    # Retrieve the most similar job's details
    most_similar_job = df.iloc[most_similar_idx]
    similarity_score = 1 - distances.flatten()[1]  # Convert distance to similarity (FAISS uses L2 distance)

    # Return the result as a dictionary
    return {
        "Job Title": most_similar_job["Job Title"],
        "Similarity Score": similarity_score,
        "Skills": most_similar_job["skills"],
    }

# Step 5: Test the function with a sample query
query_text = "Python, machine learning, data analysis"
result = get_most_similar_job(query_text)

# Display the result
print(f"Query Text: {query_text}")
print(f"Most Similar Job:")
print(f"Job Title: {result['Job Title']}")
print(f"Similarity Score: {result['Similarity Score']:.3f}")
print(f"Required Skills: {result['Skills']}")

Query Text: Python, machine learning, data analysis
Most Similar Job:
Job Title: Data Scientist
Similarity Score: 0.702
Required Skills: Machine learning algorithms Python programming Data preprocessing Deep learning Model evaluation


In [24]:
# Step 3: Define test cases (user inputs for different roles)
test_cases = {
    "Data Scientist": "Python, Machine Learning, SQL",
    "Software Engineer": "Python, Java, Software Development, Algorithms",
    "DevOps Engineer": "AWS, Kubernetes, CI/CD, Docker",
    "UX Designer": "User Experience, Wireframing, Prototyping, Figma",
    "Data Engineer": "SQL, ETL, Big Data, Apache Spark",
    "Product Manager": "Agile, Roadmapping, Stakeholder Management, Product Lifecycle"
}

# Step 4: Define a function to find the most similar job
def get_most_similar_job(query_text):
    """
    Perform a similarity search using FAISS and return the single most similar job.

    Args:
        query_text (str): The user input text (e.g., skills or job description).

    Returns:
        dict: A dictionary containing the most similar job's details and similarity score.
    """
    # Encode the query text into an embedding
    query_embedding = model.encode(query_text)
    query_embedding = query_embedding.astype(np.float32).reshape(1, -1)

    # Perform similarity search using FAISS
    distances, indices = index.search(query_embedding, k=2)  # Retrieve top 2 to exclude self-match

    # Exclude the exact match (if the query job exists in the dataset)
    most_similar_idx = indices.flatten()[1]  # Skip the first result (self-match)

    # Retrieve the most similar job's details
    most_similar_job = df.iloc[most_similar_idx]
    similarity_score = 1 - distances.flatten()[1]  # Convert distance to similarity (FAISS uses L2 distance)

    # Return the result as a dictionary
    return {
        "Job Title": most_similar_job["Job Title"],
        "Similarity Score": similarity_score,
        "Skills": most_similar_job["skills"]
    }

# Step 5: Loop through each test case and generate recommendations
for role, skills in test_cases.items():
    print(f"Testing for Role: {role}")
    print(f"Skills: {skills}")

    # Get the most similar job
    result = get_most_similar_job(skills)

    # Display the result
    print(f"Most Similar Job: {result['Job Title']}")
    print(f"Similarity Score: {result['Similarity Score']:.3f}")
    print(f"Required Skills: {result['Skills']}")
    print("-" * 50)

Testing for Role: Data Scientist
Skills: Python, Machine Learning, SQL
Most Similar Job: Data Analyst
Similarity Score: 0.747
Required Skills: Machine learning algorithms and libraries (e.g., scikit-learn, TensorFlow, PyTorch) Statistical analysis and modeling Data preprocessing and cleaning Big data technologies (e.g., Hadoop, Spark) Data visualization Strong programming skills (Python, R)
--------------------------------------------------
Testing for Role: Software Engineer
Skills: Python, Java, Software Development, Algorithms
Most Similar Job: Software Developer
Similarity Score: 0.700
Required Skills: Mobile app development languages (e.g., Java, Swift, Kotlin) Cross-platform development (e.g., React Native, Flutter) Mobile app design principles APIs and web services integration Debugging and troubleshooting
--------------------------------------------------
Testing for Role: DevOps Engineer
Skills: AWS, Kubernetes, CI/CD, Docker
Most Similar Job: Systems Engineer
Similarity Score

In [27]:
# Step 3: Define test cases (user inputs for different roles)
test_cases = {
    "Data Scientist": "Python, Machine Learning, SQL",
    "Software Engineer": "Python, Java, Software Development, Algorithms",
    "DevOps Engineer": "AWS, Kubernetes, CI/CD, Docker",
    "UX Designer": "User Experience, Wireframing, Prototyping, Figma",
    "Data Engineer": "SQL, ETL, Big Data, Apache Spark",
    "Product Manager": "Agile, Roadmapping, Stakeholder Management, Product Lifecycle"
}

# Step 4: Define a function to find the most similar job
def get_most_similar_job(query_text):
    """
    Perform a similarity search using FAISS and return the single most similar job.

    Args:
        query_text (str): The user input text (e.g., skills or job description).

    Returns:
        dict: A dictionary containing the most similar job's details and similarity score.
    """
    # Encode the query text into an embedding
    query_embedding = model.encode(query_text)
    query_embedding = query_embedding.astype(np.float32).reshape(1, -1)

    # Perform similarity search using FAISS
    distances, indices = index.search(query_embedding, k=2)  # Retrieve top 2 to exclude self-match

    # Exclude the exact match (if the query job exists in the dataset)
    most_similar_idx = indices.flatten()[1]  # Skip the first result (self-match)

    # Retrieve the most similar job's details
    most_similar_job = df.iloc[most_similar_idx]
    similarity_score = 1 - distances.flatten()[1]  # Convert distance to similarity (FAISS uses L2 distance)

    # Return the result as a dictionary
    return {
        "Job Title": most_similar_job["Job Title"],
        "Similarity Score": similarity_score,
        "Skills": most_similar_job["skills"]
    }

# Step 5: Loop through each test case and generate recommendations
for role, skills in test_cases.items():
    print(f"Role: {role}")
    print(f"Skills: {skills}")

    # Get the most similar job
    result = get_most_similar_job(skills)

    # Display the result
    # print(f"Most Similar Job: {result['Job Title']}")
    print(f"Similarity Score: {result['Similarity Score']:.3f}")
    print(f"Required Skills: {result['Skills']}")
    print("-" * 50)

Role: Data Scientist
Skills: Python, Machine Learning, SQL
Similarity Score: 0.747
Required Skills: Machine learning algorithms and libraries (e.g., scikit-learn, TensorFlow, PyTorch) Statistical analysis and modeling Data preprocessing and cleaning Big data technologies (e.g., Hadoop, Spark) Data visualization Strong programming skills (Python, R)
--------------------------------------------------
Role: Software Engineer
Skills: Python, Java, Software Development, Algorithms
Similarity Score: 0.700
Required Skills: Mobile app development languages (e.g., Java, Swift, Kotlin) Cross-platform development (e.g., React Native, Flutter) Mobile app design principles APIs and web services integration Debugging and troubleshooting
--------------------------------------------------
Role: DevOps Engineer
Skills: AWS, Kubernetes, CI/CD, Docker
Similarity Score: 0.761
Required Skills: Cloud systems engineering Cloud infrastructure (e.g., AWS, Azure) DevOps practices Automation Security in the clou

In [28]:
# Step 3: Define a function to recommend a role based on skills
def recommend_role(input_skills):
    """
    Recommends a job role based on a list of input skills.

    Args:
        input_skills (list): A list of skills provided by the user.

    Returns:
        str: The recommended job role.
    """
    # Combine the input skills into a single string
    input_text = ", ".join(input_skills)

    # Encode the input skills into an embedding
    input_embedding = model.encode(input_text)
    input_embedding = input_embedding.astype(np.float32).reshape(1, -1)

    # Perform similarity search using FAISS
    distances, indices = index.search(input_embedding, k=1)  # Retrieve top 1 match

    # Retrieve the most similar job's details
    most_similar_idx = indices.flatten()[0]
    recommended_role = df.iloc[most_similar_idx]["Job Title"]
    similarity_score = 1 - distances.flatten()[0]  # Convert distance to similarity (FAISS uses L2 distance)

    # Print the result
    print(f"Recommended Role: {recommended_role}")
    print(f"Similarity Score: {similarity_score:.3f}")
    return recommended_role

# Step 4: Test the function with sample skills
input_skills = ["Python", "SQL", "Data Analysis"]
recommended_role = recommend_role(input_skills)

Recommended Role: Database Administrator
Similarity Score: 0.695


In [30]:
import faiss
import numpy as np

# Step 1: Simulate database vectors (embeddings for job descriptions)
np.random.seed(42)  # For reproducibility
num_jobs = 1000  # Number of jobs in the database
embedding_dim = 128  # Dimensionality of embeddings
database_vectors = np.random.rand(num_jobs, embedding_dim).astype('float32')

# Normalize database vectors
database_vectors = database_vectors / np.linalg.norm(database_vectors, axis=1, keepdims=True)

# Step 2: Create FAISS index
index = faiss.IndexFlatIP(embedding_dim)  # Inner Product (cosine similarity)
index.add(database_vectors)

# Step 3: Define multiple input skills for different jobs
input_skills = {
    "Data Scientist": "Python, Machine Learning, SQL, Data Analysis",
    "Software Engineer": "Java, Python, Software Development, Algorithms, Debugging",
    "DevOps Engineer": "AWS, Kubernetes, CI/CD, Docker, Cloud Computing",
    "UX Designer": "User Experience, Wireframing, Prototyping, Figma, Design Thinking",
    "Network Engineer": "Networking, Cisco, Firewalls, TCP/IP, Security Protocols",
    "Marketing Specialist": "SEO, Social Media, Content Creation, Analytics, Copywriting",
    "Financial Analyst": "Excel, Financial Modeling, Budgeting, Forecasting, Reporting",
    "HR Manager": "Recruitment, Employee Relations, Performance Management, Training",
}

# Step 4: Simulate query vectors for each job's skills
def generate_query_vector(skill_text, embedding_dim):
    """
    Simulates an embedding for a given skill text.
    In practice, this would be replaced with a real embedding model like Sentence-BERT.
    """
    np.random.seed(hash(skill_text) % 1000000)  # Deterministic seed based on skill text
    query_vector = np.random.rand(1, embedding_dim).astype('float32')
    query_vector = query_vector / np.linalg.norm(query_vector, axis=1, keepdims=True)
    return query_vector

# Step 5: Perform similarity search for each job's skills
k = 5  # Top 5 matches
for job_title, skills in input_skills.items():
    print(f"\n🔍 Testing for Role: {job_title}")
    print(f"Skills: {skills}")

    # Generate query vector for the job's skills
    query_vector = generate_query_vector(skills, embedding_dim)

    # Perform similarity search
    distances, indices = index.search(query_vector, k)

    # Display results
    print("Top Matches:")
    for i, (distance, idx) in enumerate(zip(distances.flatten(), indices.flatten())):
        similarity_score = distance  # Inner product equals cosine similarity for normalized vectors
        print(f"  {i + 1}. Job Index: {idx}, Similarity Score: {similarity_score:.3f}")


🔍 Testing for Role: Data Scientist
Skills: Python, Machine Learning, SQL, Data Analysis
Top Matches:
  1. Job Title: 176, Similarity Score: 0.816
  2. Job Title: 814, Similarity Score: 0.810
  3. Job Title: 995, Similarity Score: 0.807
  4. Job Title: 154, Similarity Score: 0.805
  5. Job Title: 980, Similarity Score: 0.804

🔍 Testing for Role: Software Engineer
Skills: Java, Python, Software Development, Algorithms, Debugging
Top Matches:
  1. Job Title: 422, Similarity Score: 0.824
  2. Job Title: 38, Similarity Score: 0.808
  3. Job Title: 272, Similarity Score: 0.802
  4. Job Title: 304, Similarity Score: 0.801
  5. Job Title: 302, Similarity Score: 0.800

🔍 Testing for Role: DevOps Engineer
Skills: AWS, Kubernetes, CI/CD, Docker, Cloud Computing
Top Matches:
  1. Job Title: 309, Similarity Score: 0.824
  2. Job Title: 356, Similarity Score: 0.822
  3. Job Title: 281, Similarity Score: 0.820
  4. Job Title: 952, Similarity Score: 0.815
  5. Job Title: 352, Similarity Score: 0.814


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 292167 entries, 0 to 292166
Data columns (total 22 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Experience           292167 non-null  object 
 1   Qualifications       292167 non-null  object 
 2   Salary Range         292167 non-null  object 
 3   location             292167 non-null  object 
 4   Country              292167 non-null  object 
 5   latitude             292167 non-null  float64
 6   longitude            292167 non-null  float64
 7   Work Type            292167 non-null  object 
 8   Company Size         292167 non-null  int64  
 9   Job Posting Date     292167 non-null  object 
 10  Preference           292167 non-null  object 
 11  Contact              292167 non-null  object 
 12  Job Title            292167 non-null  object 
 13  Role                 292167 non-null  object 
 14  Job Description      292167 non-null  object 
 15  Benefits         